# Portfolio Allocation - FX Pairs

This notebook compares allocation methods after the equal-weight baseline. Production advances
the ten model configurations with highest validation backtest Sharpe for each label. Each carries
its best checkpoint and signal mapping into the allocation comparison. Preview mode uses a
deterministic reduced catalog selection and never writes an official population or candidate set.

**Learning objectives**

- Select configurations from an immutable equal-weight candidate set.
- Preserve the selected checkpoint and signal mapping when configurations advance to allocation.
- Change allocation while holding prediction, signal, costs, and execution fixed.

**Book reference**: Chapter 17

**Prerequisite**: `13_backtest`.

In [1]:
"""Run the FX allocation sweep from the frozen equal-weight population."""

from collections.abc import Iterable
from copy import deepcopy
from typing import Any

import polars as pl
import yaml

from case_studies.research import (
    BacktestResult,
    CandidateSet,
    OfficialPopulation,
    PredictionResult,
    Result,
    candidate_set_supersedes,
    open_study,
    plan_backtests,
    population_supersedes,
    research_name,
    run_backtests,
    strategy_warmup_periods,
    superseded_members,
)
from case_studies.utils.sweep_config import (
    get_allocators,
    get_top_n_predictions,
)
from utils.paths import get_case_study_dir
from utils.reproducibility import set_global_seeds

In [2]:
CASE_STUDY_ID = "fx_pairs"
EXECUTION_TIER = "canonical"
WORKSPACE: str = ""
LABEL = ""
SPLIT = "validation"
TOP_N_CONFIGS = 0
TOP_K = 0
TOP_N_PREDICTIONS = None
SEED = 42
RUN_SWEEP = True
FORCE_REBACKTEST = False
POPULATION_NAME = ""
BASELINE_POPULATION_NAME = None
SUPERSEDES_ALLOCATION_BACKTESTS: str = "e487eb0d75db"

# A candidate set is sealed once written, so a run whose members differ from the recorded
# generation has to name the set it replaces - the same shape 15_risk_management and 16_costs
# already carry, and keyed by the full set name because that is what the refusal prints. These
# three moved when 10a_dl_lstm registered the lstm_h64 checkpoints the training menu declares:
# the equal-weight baselines went from 1,452 to 1,572, so every label's candidate set gained
# the 40 backtests riding the new predictions. Resolved through `candidate_set_supersedes`
# rather than passed straight to `create`, because a reader's clean clone has no generation to
# supersede and `create` refuses a first version that claims to replace one.
SUPERSEDES_CANDIDATE_SETS: dict[str, str] = {
    "fx_pairs:fwd_ret_1d:equal-weight-candidates": "77b9631c7f13",
    "fx_pairs:fwd_ret_5d:equal-weight-candidates": "1c6a3826af9f",
    "fx_pairs:fwd_ret_21d:equal-weight-candidates": "db7b24f2a58b",
}

## Resolve the equal-weight inputs

Canonical execution reads the exact population frozen by the baseline notebook. Its label-specific
candidate sets provide the only performance ranking used here. The selected unit is a model
configuration. After a configuration advances, its best checkpoint and signal mapping advance.

In [3]:
set_global_seeds(SEED)
universe_symbols = yaml.safe_load(
    (get_case_study_dir(CASE_STUDY_ID) / "config" / "setup.yaml").read_text()
)["universe"]["symbols"]
n_assets = len(universe_symbols)
max_sleeve = n_assets // 2
if SPLIT != "validation":
    raise ValueError("allocation selection uses validation backtests")
if FORCE_REBACKTEST:
    raise ValueError("identical complete backtests are reused by identity")
if not RUN_SWEEP:
    raise ValueError("set RUN_SWEEP=True to execute the visible allocation request")

study = open_study(CASE_STUDY_ID, execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
# The execution tier decides which registry namespace this run reads and writes;
# the reduction knobs decide only how much of it is covered. Inferring the tier
# from the knobs conflated the two, so any reduced run went looking for preview
# predictions - and a reduced run over a canonical upstream, which is what the
# test suite exercises, then resolved no rows at all.
include_preview = EXECUTION_TIER == "preview"


def _resolve_baseline_scope(output_scope: str, input_scope: str | None) -> str:
    return output_scope if input_scope is None else input_scope


baseline_population_name = _resolve_baseline_scope(POPULATION_NAME, BASELINE_POPULATION_NAME)

# The tier decides the namespace, so a canonical run may legitimately be narrowed -
# but a narrowed run declares a different set of members than the canonical
# population does, and a population is immutable once written. Such a run must
# publish under its own name rather than register a partial snapshot of the allocation sweep
# under the canonical one.
if (
    (TOP_K or TOP_N_PREDICTIONS is not None or TOP_N_CONFIGS or LABEL)
    and not include_preview
    and not POPULATION_NAME
):
    raise ValueError(
        "this run narrows the allocation sweep, so it cannot publish the canonical "
        "population; pass POPULATION_NAME to give it its own"
    )
catalog = study.predictions.table(include_preview=include_preview).filter(
    (pl.col("identity_status") == "current")
    & (pl.col("split") == SPLIT)
    & pl.col("complete")
    & (pl.col("execution_tier") == ("preview" if include_preview else "canonical"))
)
# `identity_status` is the schema version a row was written under, not a statement about which
# generation its producer publishes. A model notebook that refits leaves the generation it
# replaced in the registry, complete and current, so this filter alone would carry a retired
# prediction set into the sweep. `superseded_members` reads the lineage instead - see
# `13_backtest`, which drops the same set before it freezes the baseline population.
# `SUPERSEDES_ALLOCATION_BACKTESTS` names the snapshot this run replaces under the name it publishes,
# offered through `population_supersedes` on the same rule. It is empty until that name has a
# first generation; after that, an upstream refit changes this population's member list and
# the registry refuses the write without it. `13_backtest` states the reasoning once.
retired = superseded_members(study, member_kind="prediction")
if retired:
    catalog = catalog.filter(~pl.col("prediction_hash").is_in(list(retired)))
if LABEL:
    catalog = catalog.filter(pl.col("label") == LABEL)
if catalog.is_empty():
    raise RuntimeError("allocation resolved no complete prediction rows")


def _result_config(result: BacktestResult) -> tuple[str, str, str]:
    training = result.lineage()["training_spec"]
    return str(training["label"]), str(training["family"]), str(training["config_name"])


def _select_configuration_survivors(
    ranked_results: Iterable[BacktestResult], limit: int
) -> list[BacktestResult]:
    """Keep the best baseline result for each distinct model configuration."""
    survivors = []
    seen: set[tuple[str, str]] = set()
    for result in ranked_results:
        config = _result_config(result)[1:]
        if config in seen:
            continue
        survivors.append(result)
        seen.add(config)
        if len(survivors) == limit:
            break
    return survivors


def _baseline_top_k(result: BacktestResult) -> int:
    signal = result.spec()["strategy"]["signal"]
    if signal.get("method") != "equal_weight_top_k" or not isinstance(signal.get("top_k"), int):
        raise RuntimeError(f"baseline {result.hash} does not carry a valid top-k signal mapping")
    return int(signal["top_k"])


def _open_backtests(hashes: Iterable[str]) -> list[BacktestResult]:
    results = [Result.open(study, value, include_preview=include_preview) for value in hashes]
    if any(not isinstance(result, BacktestResult) for result in results):
        raise TypeError("the equal-weight population contains a non-backtest result")
    return [result for result in results if isinstance(result, BacktestResult)]


def _preview_baselines(rows: pl.DataFrame) -> list[BacktestResult]:
    """The baselines 13_backtest registered, read rather than reconstructed.

    Rebuilding the identity here meant restating the upstream run's `top_k`, and the two
    notebooks reduce independently: a preview gives 13_backtest its own `TOP_K` and says
    nothing to this one, so the reconstruction was a guess about someone else's
    parameters. A wrong guess does not report a disagreement - it computes a hash that
    was never written and fails looking for it, or worse, finds an unrelated run. The
    canonical branch below already reads its members from the published population; this
    is the same read against the preview registry.
    """
    registered = study.backtests.table(include_preview=True).filter(
        (pl.col("stage") == "signal")
        & (pl.col("execution_tier") == "preview")
        & pl.col("prediction_hash").is_in(rows.get_column("prediction_hash").implode())
        & (pl.col("identity_status") == "current")
        & pl.col("complete")
    )
    if registered.is_empty():
        raise RuntimeError(
            "no preview equal-weight baselines are registered for this prediction "
            "catalog; run 13_backtest at the same reduction before this notebook"
        )
    return _open_backtests(registered.get_column("backtest_hash").unique().sort())


if include_preview:
    baseline_results = _preview_baselines(catalog)
else:
    baseline_population = OfficialPopulation.one(
        study,
        name=research_name(
            CASE_STUDY_ID,
            "equal-weight-baselines",
            scope=baseline_population_name,
        ),
    )
    baseline_population.require_complete()
    baseline_results = _open_backtests(baseline_population.members)

if any(result.registry_record()["stage"] != "signal" for result in baseline_results):
    raise RuntimeError("the allocation input contains a non-baseline backtest")

## Advance one checkpoint and mapping per configuration

Production ranks the complete baseline results for each label. The first result for a model
configuration is its best checkpoint and signal mapping by validation Sharpe. That result occupies
one declared slot and is the only member of the configuration that advances.

In [4]:
top_n = TOP_N_CONFIGS or get_top_n_predictions(CASE_STUDY_ID, "allocation")
selected_baselines: dict[str, list[BacktestResult]] = {}
candidate_sets: dict[str, CandidateSet] = {}

# The labels come from the baselines this run resolved, not from this notebook's own
# catalog. A narrowed upstream covers fewer labels than the catalog holds, and rebuilding
# the label list locally means reproducing 13_backtest's narrowing by convention - the
# same guess that reading the registered population exists to avoid. When the run is not
# scoped it is publishing canonical names, and then the two must agree exactly.
baseline_labels = sorted({_result_config(result)[0] for result in baseline_results})
if not baseline_labels:
    raise RuntimeError("the equal-weight baselines carry no labels")
if not POPULATION_NAME and baseline_labels != sorted(catalog.get_column("label").unique()):
    raise RuntimeError(
        "the canonical baseline population does not cover every label in the catalog: "
        f"baselines {baseline_labels}, catalog {sorted(catalog.get_column('label').unique())}"
    )

for label in baseline_labels:
    label_results = [result for result in baseline_results if _result_config(result)[0] == label]
    if include_preview:
        ranked_results = sorted(
            label_results,
            key=lambda result: (*_result_config(result)[1:], result.hash),
        )
    else:
        _set_name = research_name(
            CASE_STUDY_ID, f"{label}:equal-weight-candidates", scope=POPULATION_NAME
        )
        candidates = CandidateSet.create(
            study,
            name=_set_name,
            members=label_results,
            supersedes=candidate_set_supersedes(
                study, name=_set_name, declared=SUPERSEDES_CANDIDATE_SETS.get(_set_name)
            ),
        )
        candidate_sets[label] = candidates
        ranked_results = list(candidates.ranked_validation_sharpe())
        if any(not isinstance(result, BacktestResult) for result in ranked_results):
            raise TypeError("validation-Sharpe ranking returned a non-backtest result")
    selected_baselines[label] = _select_configuration_survivors(ranked_results, top_n)
    available_configs = {_result_config(result)[1:] for result in label_results}
    if len(selected_baselines[label]) != min(top_n, len(available_configs)):
        raise RuntimeError(f"configuration selection for {label} is incomplete")

baseline_predictions = {result.registry_record()["prediction_hash"] for result in baseline_results}
if not POPULATION_NAME:
    missing = set(catalog.get_column("prediction_hash")) - baseline_predictions
    if missing:
        raise RuntimeError(
            "the canonical baseline population does not cover every prediction in the "
            f"catalog: {len(missing)} uncovered"
        )
selected_rows = []
for label, survivors in selected_baselines.items():
    for baseline in survivors:
        prediction_hash = baseline.registry_record()["prediction_hash"]
        member = catalog.filter(pl.col("prediction_hash") == prediction_hash)
        if member.height != 1:
            raise RuntimeError(
                f"selected baseline {baseline.hash} resolved to {member.height} prediction rows"
            )
        selected_rows.append(member.with_columns(pl.lit(_baseline_top_k(baseline)).alias("top_k")))

selected = pl.concat(selected_rows).unique(subset=["prediction_hash"], maintain_order=True)
if selected.get_column("prediction_hash").n_unique() != selected.height:
    raise RuntimeError("allocation input contains duplicate prediction identities")
selected.select(
    "label",
    "family",
    "config_name",
    "checkpoint_kind",
    "checkpoint_value",
    "top_k",
    "prediction_hash",
).sort("label", "family", "config_name", "checkpoint_value")

label,family,config_name,checkpoint_kind,checkpoint_value,top_k,prediction_hash
str,str,str,str,i64,i32,str
"""fwd_ret_1d""","""deep_learning""","""lstm_h64""","""epoch""",30,10,"""476261e05c77"""
"""fwd_ret_1d""","""deep_learning""","""tcn""","""epoch""",5,5,"""56421df10fc4"""
"""fwd_ret_1d""","""linear""","""enet_f0.5""","""final""",null,10,"""46143f6f7514"""
"""fwd_ret_1d""","""linear""","""enet_f0.7""","""final""",null,10,"""eef01e484225"""
"""fwd_ret_1d""","""linear""","""enet_f0.85""","""final""",null,5,"""8e7ef1ec8ac7"""
…,…,…,…,…,…,…
"""fwd_ret_5d""","""gbm""","""leaves_7_huber""","""iteration""",50,10,"""29f4446969d4"""
"""fwd_ret_5d""","""gbm""","""leaves_7_mae""","""iteration""",50,10,"""12937b00c5a3"""
"""fwd_ret_5d""","""linear""","""enet_f0.85""","""final""",null,10,"""25b331fcc044"""


## Plan and freeze the allocation grid

Each request changes only the allocator. The selected prediction and `top_k` mapping pass directly
from the winning baseline result to the shared backtest boundary. Production freezes every
expected identity before the first allocation result is written.

In [5]:
allocators = get_allocators(CASE_STUDY_ID)
if not allocators or any(config.get("method") == "equal_weight" for config in allocators):
    raise RuntimeError("allocation methods must be non-empty and exclude the equal-weight baseline")

jobs: list[dict[str, Any]] = []
if TOP_K and selected.filter(pl.col("top_k") != TOP_K).height:
    raise RuntimeError("TOP_K differs from the mapping selected by the upstream baseline")
for label, top_k in selected.select("label", "top_k").unique().sort("label", "top_k").iter_rows():
    rows = selected.filter((pl.col("label") == label) & (pl.col("top_k") == top_k)).drop("top_k")
    if top_k > max_sleeve:
        raise RuntimeError(
            f"selected top_k {top_k} exceeds the {max_sleeve}-pair sleeve ceiling for a "
            f"long-short account on {n_assets} pairs"
        )
    for allocation in allocators:
        jobs.append(
            {
                "label": label,
                "top_k": top_k,
                "allocation": allocation,
                "predictions": rows,
                "expected": rows.height,
            }
        )

pl.DataFrame(
    [
        {
            "label": job["label"],
            "top_k": job["top_k"],
            "allocator": job["allocation"]["method"],
            "prediction_sets": job["expected"],
        }
        for job in jobs
    ]
)

label,top_k,allocator,prediction_sets
str,i64,str,i64
"""fwd_ret_1d""",5,"""score_weighted""",3
"""fwd_ret_1d""",5,"""inverse_vol""",3
"""fwd_ret_1d""",5,"""risk_parity""",3
"""fwd_ret_1d""",5,"""mvo_ledoit_wolf""",3
"""fwd_ret_1d""",5,"""hrp""",3
…,…,…,…
"""fwd_ret_5d""",10,"""inverse_vol""",9
"""fwd_ret_5d""",10,"""risk_parity""",9
"""fwd_ret_5d""",10,"""mvo_ledoit_wolf""",9


In [6]:
planned_hashes = []
for job in jobs:
    plan = plan_backtests(
        study,
        predictions=job["predictions"],
        signal={"method": "equal_weight_top_k", "top_k": job["top_k"]},
        allocation=job["allocation"],
        chapter="17",
    )
    if len(plan.members) != job["expected"]:
        raise RuntimeError("an allocation plan omitted a selected prediction")
    planned_hashes.extend(plan.expected_hashes)
if len(planned_hashes) != len(set(planned_hashes)):
    raise RuntimeError("two planned allocation requests collapse to the same identity")

allocation_population = None
if not include_preview:
    allocations_name = research_name(CASE_STUDY_ID, "allocation-backtests", scope=POPULATION_NAME)
    allocation_population = OfficialPopulation.create(
        study,
        name=allocations_name,
        member_kind="backtest",
        members=planned_hashes,
        supersedes=population_supersedes(
            study, name=allocations_name, declared=SUPERSEDES_ALLOCATION_BACKTESTS
        ),
    )
    print(f"Frozen expected allocation population: {allocation_population.hash}")

Frozen expected allocation population: 8b94efd2b615


## Execute the frozen allocation grid, then validate what was frozen

The population is validated in the cell that fills it, because the two are one act: the expected
set was written down before the first member ran, and `require_complete` is what turns that
declaration into a published result.

In [7]:
# A sweep that recomputes everything and a sweep that recomputes nothing print the same summary
# unless the two are counted apart. `run_backtests` serves an identity that is already registered
# and complete instead of running it again, which is what makes a re-run affordable and what makes
# a bare member count say nothing about whether this run did any work.
#
# The runner already knows which it did and says so per member in `execution.diagnostics`, as
# `status` "reused" or "completed". Comparing against the registered hashes instead would be
# wrong in both directions: a registered-but-partial backtest is in that set, gets recomputed and
# would report as reused, and a preview re-run reads a table that excludes preview rows by default
# and would report every reused member as computed.
run_status: list[str] = []
allocation_results = []
for job in jobs:
    execution = run_backtests(
        study,
        predictions=job["predictions"],
        signal={"method": "equal_weight_top_k", "top_k": job["top_k"]},
        allocation=job["allocation"],
        chapter="17",
    )
    if len(execution.results) != job["expected"]:
        raise RuntimeError("an allocation member disappeared during execution")
    allocation_results.extend(execution.results)
    run_status.extend(entry["status"] for entry in execution.diagnostics)

expected_count = sum(job["expected"] for job in jobs)
if len(allocation_results) != expected_count:
    raise RuntimeError(
        f"expected {expected_count} allocation runs, found {len(allocation_results)}"
    )
if {result.hash for result in allocation_results} != set(planned_hashes):
    raise RuntimeError("completed allocation identities differ from the frozen plan")
if any(
    not result.complete or result.registry_record()["stage"] != "allocation"
    for result in allocation_results
):
    raise RuntimeError("the allocation population is incomplete or misclassified")

served = run_status.count("reused")
print(
    f"Allocation backtests: {len(allocation_results) - served} computed, {served} served from the registry, "
    f"{len(allocation_results)} in the population"
)


def _leaves(value: Any, prefix: str = "") -> dict[str, Any]:
    """Flatten a nested spec to dotted paths so a mismatch can name what moved."""
    if isinstance(value, dict):
        flat: dict[str, Any] = {}
        for key, item in value.items():
            flat.update(_leaves(item, f"{prefix}.{key}" if prefix else str(key)))
        return flat
    return {prefix: value}


def _non_allocation_projection(spec: dict[str, Any], *, drop_prices: bool) -> dict[str, Any]:
    projected = deepcopy(spec)
    projected.pop("chapter", None)
    projected.pop("_runtime_backtest_config", None)
    projected.get("strategy", {}).pop("allocation", None)
    metadata = projected.get("backtest_config", {}).get("metadata")
    if isinstance(metadata, dict):
        metadata.pop("chapter", None)
    if drop_prices:
        projected.get("input_identity", {}).pop("prices", None)
    return projected


for result in allocation_results:
    prediction_hash = result.registry_record()["prediction_hash"]
    signal = result.spec()["strategy"]["signal"]
    siblings = [
        baseline
        for baseline in baseline_results
        if baseline.registry_record()["prediction_hash"] == prediction_hash
        and baseline.spec()["strategy"]["signal"] == signal
    ]
    if len(siblings) != 1:
        raise RuntimeError(
            f"allocation {result.hash} resolved to {len(siblings)} equal-weight siblings"
        )
    # input_identity.prices is a function of the allocation, not something held constant
    # across it. A moment allocator declares a warmup, load_backtest_prices_for leaves the
    # start of the load window unconstrained by that many periods, and the frame it returns
    # digests differently from the equal-weight baseline's - by design, with the extra
    # prefix consumed by the rolling window and excluded from return aggregation.
    #
    # The decision is taken once, from the allocation, and applied to both sides. Asking
    # each spec about its own allocation drops the key from the allocation projection and
    # keeps it in the baseline's, so the two differ on the key's presence rather than its
    # value - a comparison made unequal by the very step meant to make it fair.
    #
    # Where the allocator declares no warmup both sides load the same window, and the
    # digest is a real check that the allocation did not move the price input.
    drop_prices = bool(
        strategy_warmup_periods({"allocation": result.spec()["strategy"].get("allocation")})
        if result.spec()["strategy"].get("allocation")
        else 0
    )
    allocation_projection = _non_allocation_projection(result.spec(), drop_prices=drop_prices)
    baseline_projection = _non_allocation_projection(siblings[0].spec(), drop_prices=drop_prices)
    if allocation_projection != baseline_projection:
        # Naming the fields is the difference between a check and a diagnosis. The
        # projections are nested, so compare the flattened leaves and report only the
        # paths that actually disagree.
        divergent = sorted(
            path
            for path in set(_leaves(allocation_projection)) | set(_leaves(baseline_projection))
            if _leaves(allocation_projection).get(path) != _leaves(baseline_projection).get(path)
        )
        raise RuntimeError(
            f"allocation {result.hash} changed a non-allocation strategy field against "
            f"baseline {siblings[0].hash}: "
            + "; ".join(
                f"{path}: baseline={_leaves(baseline_projection).get(path)!r} "
                f"allocation={_leaves(allocation_projection).get(path)!r}"
                for path in divergent
            )
        )

if not include_preview:
    if allocation_population is None:
        raise RuntimeError("the canonical allocation population was not frozen before execution")
    allocation_population.require_complete()
    print(f"Official allocation population: {allocation_population.hash}")
else:
    print("Preview allocation results remain outside official populations and candidate sets.")

~/ml4t/public-fx-lstm/case_studies/utils/conformal.py:471: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  targets.join_asof(


Allocation backtests: 6 computed, 174 served from the registry, 180 in the population


Official allocation population: 8b94efd2b615


## Key takeaways

- Validation Sharpe ranks an immutable equal-weight candidate set for each label.
- Each advancing configuration retains its best baseline checkpoint and signal mapping.
- Preview reductions exercise the same backtest engine without entering production populations.